In [1]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Original Problematic Cases

In [2]:
# Paths
base_dir = Path("/Users/ce12/Documents/GitHub/cellsem-agent/cellsem_agent/graphs/cxg_annotate")
original_dir = base_dir / "resources/input/less_specific_to_rerun"
rerun_output_dir = base_dir / "resources/output/output_tissue_context"  # Will contain new results

# Load all original problematic cases
original_cases = []
for tsv_file in original_dir.glob("*.tsv"):
    df = pd.read_csv(tsv_file, sep="\t")
    df['original_dataset'] = tsv_file.stem
    original_cases.append(df)

original_df = pd.concat(original_cases, ignore_index=True)
print(f"Total problematic cases: {len(original_df)}")
print(f"\nBreakdown by error type:")
print(original_df['error_type'].value_counts())

Total problematic cases: 147

Breakdown by error type:
error_type
less_specific    92
other            55
Name: count, dtype: int64


## 2. Load New Results (After Tissue Context)

In [3]:
# Load new grounding results
# NOTE: Run this cell AFTER you've run AMICA with tissue context on the clean files

new_results = []
for dataset_folder in rerun_output_dir.iterdir():
    if dataset_folder.is_dir():
        groundings_file = dataset_folder / "groundings.tsv"
        if groundings_file.exists():
            df = pd.read_csv(groundings_file, sep="\t")
            df['dataset_name'] = dataset_folder.name
            new_results.append(df)

if new_results:
    new_df = pd.concat(new_results, ignore_index=True)
    print(f"Total new results: {len(new_df)}")
    print(f"\nColumns: {new_df.columns.tolist()}")
else:
    print("No new results found yet. Run AMICA first!")
    new_df = pd.DataFrame()

Total new results: 147

Columns: ['annotation_text', 'cl_id', 'cl_label', 'article_id_doi', 'dataset_name', 'enrichment', 'grounding_cl_id', 'grounding_cl_label', 'result']


## 3. Compare Results

In [5]:
if not new_df.empty:
    # Merge original and new results
    # Match on annotation_text and cl_id (author's ground truth)
    comparison = original_df.merge(
        new_df[['annotation_text', 'cl_id', 'grounding_cl_id', 'grounding_cl_label', 'enrichment']],
        on=['annotation_text', 'cl_id'],
        suffixes=('_original', '_new'),
        how='left'
    )
    
    # Determine if the new result is an improvement
    comparison['improved'] = comparison['cl_id'] == comparison['grounding_cl_id']
    
    print(f"\nImprovement Summary:")
    print(f"Total cases analyzed: {len(comparison)}")
    print(f"Cases with new results: {comparison['grounding_cl_id'].notna().sum()}")
    print(f"Improved (now correct): {comparison['improved'].sum()}")
    print(f"Still incorrect: {(~comparison['improved'] & comparison['grounding_cl_id'].notna()).sum()}")
    print(f"No grounding found: {comparison['grounding_cl_id'].isna().sum()}")
    
    print(f"\nImprovement by error type:")
    improvement_by_type = comparison.groupby('error_type')['improved'].agg(['sum', 'count'])
    improvement_by_type['percentage'] = (improvement_by_type['sum'] / improvement_by_type['count'] * 100).round(2)
    print(improvement_by_type)
else:
    print("Waiting for new results...")

KeyError: 'grounding_cl_id'

## 4. Visualize Improvements

In [ ]:
if not new_df.empty:
    # Create visualizations
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Chart 1: Improvement by error type
    improvement_counts = comparison.groupby(['error_type', 'improved']).size().unstack(fill_value=0)
    improvement_counts.plot(kind='bar', ax=axes[0], color=['#ff6b6b', '#51cf66'])
    axes[0].set_title('Improvement by Error Type', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Error Type')
    axes[0].set_ylabel('Count')
    axes[0].legend(['Still Incorrect', 'Improved'], title='Status')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Chart 2: Top datasets with improvements
    dataset_improvements = comparison.groupby('original_dataset')['improved'].sum().sort_values(ascending=False).head(10)
    dataset_improvements.plot(kind='barh', ax=axes[1], color='#4c6ef5')
    axes[1].set_title('Top 10 Datasets with Most Improvements', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Number of Improvements')
    axes[1].set_ylabel('Dataset')
    
    plt.tight_layout()
    plt.show()
else:
    print("Waiting for new results...")

## 5. Detailed Case Analysis

In [ ]:
if not new_df.empty:
    # Show specific examples of improvements
    print("=" * 80)
    print("IMPROVED CASES (Sample)")
    print("=" * 80)
    
    improved_cases = comparison[comparison['improved'] == True].head(10)
    for idx, row in improved_cases.iterrows():
        print(f"\nAnnotation: {row['annotation_text']}")
        print(f"  Ground Truth: {row['CL_label']} ({row['cl_id']})")
        print(f"  Original Agent Result: {row['grounding_cl_label_original']} ({row['grounding_cl_id_original']})")
        print(f"  New Agent Result: {row['grounding_cl_label']} ({row['grounding_cl_id']}) ✓")
        if pd.notna(row['enrichment']):
            import ast
            try:
                enrichment = ast.literal_eval(row['enrichment'])
                if isinstance(enrichment, dict) and enrichment.get('tissue_context'):
                    print(f"  Tissue Context: {enrichment['tissue_context']}")
            except:
                pass
        print("-" * 80)
else:
    print("Waiting for new results...")

## 6. Cases Still Needing Improvement

In [ ]:
if not new_df.empty:
    print("=" * 80)
    print("CASES STILL INCORRECT (Sample)")
    print("=" * 80)
    
    still_wrong = comparison[(comparison['improved'] == False) & (comparison['grounding_cl_id'].notna())].head(10)
    for idx, row in still_wrong.iterrows():
        print(f"\nAnnotation: {row['annotation_text']}")
        print(f"  Ground Truth: {row['CL_label']} ({row['cl_id']})")
        print(f"  Original Agent Result: {row['grounding_cl_label_original']} ({row['grounding_cl_id_original']})")
        print(f"  New Agent Result: {row['grounding_cl_label']} ({row['grounding_cl_id']}) ✗")
        print(f"  Error Type: {row['error_type']}")
        print("-" * 80)
else:
    print("Waiting for new results...")

## 7. Export Comparison Report

In [ ]:
if not new_df.empty:
    # Save detailed comparison to file
    output_file = base_dir / "resources/output/reports/tissue_context_comparison.tsv"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(output_file, sep='\t', index=False)
    print(f"Comparison report saved to: {output_file}")
    
    # Create summary statistics
    summary = {
        'Total Cases': len(comparison),
        'Improved': comparison['improved'].sum(),
        'Still Incorrect': (~comparison['improved'] & comparison['grounding_cl_id'].notna()).sum(),
        'No Grounding': comparison['grounding_cl_id'].isna().sum(),
        'Improvement Rate (%)': (comparison['improved'].sum() / len(comparison) * 100).round(2),
        'Less Specific Improved': comparison[comparison['error_type'] == 'less_specific']['improved'].sum(),
        'Other Improved': comparison[comparison['error_type'] == 'other']['improved'].sum(),
    }
    
    summary_df = pd.DataFrame([summary])
    summary_file = base_dir / "resources/output/reports/tissue_context_summary.tsv"
    summary_df.to_csv(summary_file, sep='\t', index=False)
    print(f"Summary report saved to: {summary_file}")
    
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    for key, value in summary.items():
        print(f"{key}: {value}")
else:
    print("Waiting for new results...")